# 💬 生成式AI應用開發｜第05週

## 對話狀態、Streaming 與聊天機器人實作

本週延續第 4 週的 prompt 函式，進一步讓模型記得前文、逐步顯示回覆，並完成可重設、可觀察成本的簡易聊天機器人。


> **學生版**：三組課堂練習保留 TODO 骨架；請先完成多輪與 Streaming 示範，再實作自己的聊天 session。


## 🎯 本週學習目標

| # | 完成本週後能做到 | 對應後續課程／應用 |
|---|---|---|
| 1 | 解釋無狀態 API 與對話狀態的差異 | 除錯多輪問答、設計聊天流程 |
| 2 | 使用 `previous_response_id` 串接多輪回覆 | 快速建立原型與客服對話 |
| 3 | 用手動 history 明確管理訊息 | 儲存、裁切、稽核與跨平台整合 |
| 4 | 觀察 context 規模並設計記憶視窗 | 第 6 週成本控制與長對話管理 |
| 5 | 處理 Streaming 事件與文字增量 | 即時 UI、逐字回覆與進度顯示 |
| 6 | 完成具錯誤處理的 CLI 聊天機器人 | 後續 Web／App 聊天介面基礎 |

## 🗺️ 三堂課（150 分鐘）實作流程

| 時間 | 主題 | 實作成果 |
|---:|---|---|
| 0:00–0:20 | 無狀態請求與多輪對話 | 觀察獨立請求為何忘記前文 |
| 0:20–0:50 | `previous_response_id` 與 history | 完成兩種狀態管理方式 |
| 0:50–1:15 | 最小聊天 session | 用少量程式保存上一輪狀態 |
| 1:15–1:45 | 可重設 session 與 usage | 建立較完整的聊天物件 |
| 1:45–2:15 | Streaming events | 累積並即時顯示文字片段 |
| 2:15–2:30 | 必做練習、測試與回顧 | 完成練習 A/B，練習 C 作為進階選做 |

---

## 1️⃣ 環境準備

沿用前兩週設定：API key 放在 Colab Secrets 的 `OPENAI_API_KEY`。本週會產生多次 API 請求，請先確認模型與金鑰，再依序執行示範。


In [ ]:
# 安裝或更新 OpenAI Python SDK
!pip install -q --upgrade openai


In [ ]:
import os
from openai import OpenAI

# Colab 優先讀取 Secrets；本機執行時則改用既有環境變數。

try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    # 非 Colab 環境會走到這裡，不需要中斷 notebook。
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")

# API key 不存在時提早停止，避免後面每個 API cell 都出現難讀的錯誤。
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "找不到 OPENAI_API_KEY。請先設定 Colab Secret 或本機環境變數，再重新執行此 cell。"
    )

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI()
print("本週模型：", MODEL)
print("API key 已設定（不顯示內容）")


In [ ]:
CHAT_INSTRUCTIONS = """
# Identity
你是生成式 AI 應用開發課程的助教。
# Instructions
- 使用繁體中文，先直接回答，再補充必要說明。
- 回答控制在 150 字內；不知道時清楚說明限制。
- 不要聲稱記得目前對話以外的資料。
"""


def create_response(user_input, previous_response_id=None, store=True):
    """
    建立一個一般 Responses API 回應，並可選擇串接上一輪對話。

    第 5 週的核心觀念是「模型本身不會自動記得 notebook 前面發生的事」。
    如果要讓下一輪延續上一輪，可以傳入 previous_response_id；如果不傳，
    這次請求就是一個獨立問題。instructions 每一輪都明確重送，則是為了讓角色與回答規則穩定。

    參數 (Args):
        user_input:            使用者本輪輸入的文字，必須是非空白字串。
        previous_response_id:  上一輪 response 的 id；有值時會延續該輪上下文。
        store:                 是否讓平台保存這次 response，以便之後用 previous_response_id 串接。

    回傳 (Returns):
        OpenAI Responses API 的 response 物件；常用欄位包含 id、output_text 與 usage。

    可能拋出 (Raises):
        ValueError: user_input 不是可送出的文字。
        RuntimeError: API 呼叫失敗，並保留原始錯誤作為原因。
    """

    # 情況一：輸入文字不合法
    # 先在本機擋掉空白輸入，可以避免 API 回傳較不直觀的錯誤。
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("user_input 必須是非空白字串。")

    # 先把必要欄位集中成 request dict，後面才依情況補上對話串接欄位。
    request = {
        "model": MODEL,
        "instructions": CHAT_INSTRUCTIONS,
        "input": user_input,
        "store": store,
    }
    # 情況二：需要延續上一輪
    # previous_response_id 讓 Responses API 延續指定回合的上下文；沒有它時，本輪就是獨立請求。
    if previous_response_id:
        request["previous_response_id"] = previous_response_id

    try:
        return client.responses.create(**request)
    except Exception as exc:
        # 統一包成 RuntimeError，讓呼叫端看到比較接近課堂語境的錯誤訊息。
        raise RuntimeError(f"OpenAI API 呼叫失敗：{exc}") from exc


<div style="border-left:5px solid #f9ab00;padding:10px 14px;background:#fff8e1;">
<font color="#b06000"><b>成本與資料提醒</b></font>：每一輪都是一次 API 請求。即使用 `previous_response_id`，鏈中先前輸入仍會計入 input tokens。Responses 預設可能保存回應以支援串接；課堂資料不得包含個資、成績、密碼或其他敏感資訊。
</div>


---

## 2️⃣ 先看「無狀態」：兩次獨立請求

模型不會因為兩行 Python 程式相鄰，就自動把兩次獨立 API 呼叫視為同一段對話。下面第二次請求沒有附上第一次的內容。


In [ ]:
# 兩次都沒有傳入 previous_response_id，所以第二輪看不到第一輪內容。
standalone_1 = create_response("請記住：我叫小安，專題主題是校園活動推薦。")
print("第一輪：", standalone_1.output_text)

standalone_2 = create_response("我叫什麼名字？專題主題是什麼？")
print("第二輪：", standalone_2.output_text)


### ✍️ 觀察紀錄

- 第二輪是否正確說出姓名與主題？
- 如果碰巧答對，答案是否真的來自第一輪，還是模型猜測？
- 應用程式需要保存什麼，才能可靠延續對話？

不要以一次看似正確的輸出判斷「有記憶」；必須檢查下一輪請求實際包含的狀態。


---

## 3️⃣ 三種對話狀態管理方式

| 方式 | 應用程式保存什麼 | 適合情境 | 注意事項 |
|---|---|---|---|
| `previous_response_id` | 上一個 response ID | 同一條短期對話串 | 每輪仍重送 instructions；歷史 token 仍計費 |
| 手動 history | `user`／`assistant` 訊息 | 需要自行裁切、檢查或不保存 Response | 應用程式負責順序與內容 |
| Conversations API | durable conversation ID | 跨工作、裝置或長期 session | 保存期限與隱私策略需另行設計 |

本週實作前兩種；Conversations API 先建立概念，不作為必要練習。


In [ ]:
turn_1 = create_response(
    "請記住：我叫小安，專題主題是校園活動推薦。",
    store=True,
)
print("第一輪：", turn_1.output_text)

# 把第一輪 response id 傳給第二輪，模型才有機會沿用上一輪脈絡。
turn_2 = create_response(
    "我叫什麼名字？專題主題是什麼？",
    previous_response_id=turn_1.id,
    store=True,
)
print("第二輪：", turn_2.output_text)
print("串接 ID：", turn_1.id, "→", turn_2.id)


### 🔗 `previous_response_id` 的心智模型

```text
使用者第 1 輪 → Response A（id=A）
                         ↓ previous_response_id=A
使用者第 2 輪 → Response B（id=B）
                         ↓ previous_response_id=B
使用者第 3 輪 → Response C
```

`instructions` 只套用在目前這次 response，因此聊天程式應在每一輪明確傳入，不要假設上一輪 instructions 會自動存在。


---

## 4️⃣ 手動管理 conversation history

手動 history 會把需要的 `user` 與 `assistant` 訊息放進下一次 `input`。優點是可檢查、裁切與保存；代價是應用程式必須自行維護順序和內容。

> 進階提醒：若使用需要保留推理項目的模型，正式無狀態流程應依官方文件保存完整 `response.output` 與必要的 encrypted reasoning items。本週先用文字訊息理解基本結構。


In [ ]:
# 方法 B：手動保存訊息 history（結構透明、容易裁切與稽核）
manual_history = [
    {"role": "user", "content": "我叫小華，請記住。"},
]

manual_1 = client.responses.create(
    model=MODEL,
    input=manual_history,
    store=False,  # 手動 history 已保存上下文，因此不需要由平台保存這次 response。
)
print("第一輪：", manual_1.output_text)

# 手動 history 必須同時加入 assistant 回答與下一次 user 問題。
manual_history.extend([
    {"role": "assistant", "content": manual_1.output_text},
    {"role": "user", "content": "我叫什麼名字？"},
])

manual_2 = client.responses.create(
    model=MODEL,
    input=manual_history,
    store=False,
)
print("第二輪：", manual_2.output_text)


def history_size(messages):
    """
    用訊息數與字元數快速觀察手動 history 的規模。

    這個函式不是精準 token 計算器，而是課堂上快速觀察 context 是否變大的輔助工具。
    真正的計費與模型限制仍要看 API 回傳的 usage；但在呼叫 API 前，
    先看訊息數與字元數可以幫學生理解「手動 history 越長，送入模型的內容越多」。

    參數 (Args):
        messages: Responses API input 格式的 message list，每筆通常有 role 與 content。

    回傳 (Returns):
        dict，包含訊息數與總字元數。
    """

    # 情況一：message 有 content 欄位
    # 取出 content 後轉成字串計算長度，避免內容不是字串時直接出錯。
    chars = sum(len(str(message.get("content", ""))) for message in messages)

    # 情況二：message 沒有 content 欄位
    # message.get("content", "") 會讓缺漏內容以空字串計算，方便做粗略估算。
    return {"訊息數": len(messages), "總字元數": chars}


print("目前 history 規模：", history_size(manual_history))


### ⚖️ 選擇方式

- 想快速建立同一條短期對話：先用 `previous_response_id`。
- 想自行刪除舊訊息、插入摘要或檢查送出的內容：使用手動 history。
- 想跨 session 保存長期狀態：再評估 Conversations API 或自己的資料庫。

「模型記憶」通常是應用程式把適當資料重新提供給模型，而不是模型永久記住某位使用者。


---

## 5️⃣ 先做一個最小聊天 Session

先用最少的程式保存 `previous_response_id`。這個版本只處理「能不能接續上一輪」，還不處理 transcript、usage、reset 與錯誤保護；下一節會把它擴充成較完整的 class。

In [ ]:
class MinimalChatSession:
    """
    只保存 previous_response_id 的最小聊天 Session。

    這個類別示範「最少狀態」也能完成多輪對話：本地端只記住上一輪 response id，
    對話內容則交給平台端用 store=True 保存。優點是程式簡單；限制是本地端看不到完整 transcript，
    也不容易做自訂裁切、稽核或離線保存。
    """

    def __init__(self, instructions=CHAT_INSTRUCTIONS):
        """
        初始化最小 Session 狀態。

        參數 (Args):
            instructions: 每一輪要送給模型的角色與回答規則。
        """
        self.instructions = instructions
        # 只保存上一輪 response id，是最小化的狀態管理方式。
        self.previous_response_id = None

    def ask(self, user_text):
        """
        送出一輪對話，成功後更新 previous_response_id。

        參數 (Args):
            user_text: 使用者本輪輸入。

        回傳 (Returns):
            模型回覆文字 output_text。
        """
        request = {
            "model": MODEL,
            "instructions": self.instructions,
            "input": user_text,
            "store": True,
        }
        # 情況一：已有上一輪 id
        # 把 previous_response_id 放入 request，模型才會沿用前一輪上下文。
        if self.previous_response_id:
            request["previous_response_id"] = self.previous_response_id

        # 情況二：第一次發問
        # previous_response_id 仍是 None，request 不會加入串接欄位，這輪會成為對話起點。
        response = client.responses.create(**request)
        # API 成功後才更新 id，避免失敗時把 session 狀態改壞。
        self.previous_response_id = response.id
        return response.output_text


minimal_chat = MinimalChatSession()
RUN_MINIMAL_SESSION_DEMO = False
if RUN_MINIMAL_SESSION_DEMO:
    print(minimal_chat.ask("我正在設計課程 FAQ 聊天機器人，請記住。"))
    print(minimal_chat.ask("請重述我正在設計什麼。"))
else:
    print("最小聊天 session 已建立；確認 API 成本後，可將 RUN_MINIMAL_SESSION_DEMO 改為 True。")


---

## 6️⃣ 建立可重設的非串流 Chat Session

延續上一節的最小版本，完整 session 至少要保存：上一個 response ID、畫面要顯示的 transcript、累積 usage，以及 reset 行為。對話失敗時，不應把失敗的回合寫入狀態。

In [ ]:
class ResponseChatSession:
    """
    同時保存 response 串接狀態、可讀 transcript 與 token usage 的聊天 Session。

    相較於 MinimalChatSession，這個類別多保存 transcript 與 usage，
    適合正式 App 或課堂觀察：transcript 讓使用者看得到對話紀錄，usage 讓開發者追蹤成本，
    previous_response_id 則負責把下一輪接到上一輪。
    """

    def __init__(self, instructions=CHAT_INSTRUCTIONS):
        """
        建立聊天 Session 並初始化狀態。

        參數 (Args):
            instructions: 每一輪固定送出的角色與回答規則。
        """
        self.instructions = instructions
        self.reset()

    def reset(self):
        """
        清除本地端聊天狀態。

        reset 會清掉 previous_response_id、transcript 與 usage，讓下一次 ask() 從新對話開始。
        它不會刪除平台端已建立的 response；這點要和本地狀態重設分開理解。
        """
        # reset 只清除本地狀態，不會刪除平台端已建立的 response。
        self.previous_response_id = None
        self.transcript = []
        self.usage = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}

    def _add_usage(self, response):
        """
        將本輪 response 的 token usage 累加到 Session 統計中。

        參數 (Args):
            response: OpenAI Responses API 回傳的 response 物件。

        回傳 (Returns):
            無。此方法會直接更新 self.usage。
        """

        # 情況一：response 有 usage
        # 正常成功回應通常會包含 input/output/total token 數，可以安全累加。
        if response.usage:
            self.usage["input_tokens"] += response.usage.input_tokens
            self.usage["output_tokens"] += response.usage.output_tokens
            self.usage["total_tokens"] += response.usage.total_tokens

        # 情況二：response 沒有 usage
        # 某些錯誤或特殊回應可能沒有 usage，這時保持原本統計即可。

    def ask(self, user_text):
        """
        送出一輪一般非串流聊天，成功後更新所有 Session 狀態。

        參數 (Args):
            user_text: 使用者本輪輸入。

        回傳 (Returns):
            模型回覆文字 output_text。

        可能拋出 (Raises):
            ValueError: user_text 是空白或非字串。
            RuntimeError: API 呼叫失敗；失敗時不更新 Session 狀態。
        """

        # 情況一：輸入不合法
        if not isinstance(user_text, str) or not user_text.strip():
            raise ValueError("user_text 必須是非空白字串。")

        request = {
            "model": MODEL,
            "instructions": self.instructions,
            "input": user_text,
            "store": True,
        }
        # 情況二：已有上一輪 response id，可以延續平台端上下文。
        if self.previous_response_id:
            request["previous_response_id"] = self.previous_response_id

        try:
            response = client.responses.create(**request)
        except Exception as exc:
            # 請求失敗時不更新 previous_response_id 或 transcript，方便重試同一輪。
            raise RuntimeError(f"聊天請求失敗，狀態未更新：{exc}") from exc

        # 成功後才更新三種狀態：串接 id、可讀 transcript、token usage。
        self.previous_response_id = response.id
        self.transcript.extend([
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": response.output_text},
        ])
        self._add_usage(response)
        return response.output_text


chat = ResponseChatSession()
print(chat.ask("我正在做課程 FAQ 聊天機器人。請記住這個主題。"))
print(chat.ask("請用一句話重述我的主題。"))
print("累積 usage：", chat.usage)


### 🧠 簡易記憶不等於無限 history

長對話會遇到三個問題：

1. 舊內容逐漸增加輸入 token 與成本。
2. 過時資訊可能干擾目前任務。
3. 對話可能接近模型 context window。

常見策略包括：只留最近 N 輪、把重要事實另外保存、將舊對話摘要、在長流程使用 compaction。第 5 週先完成最近 N 輪與重要事實的簡化版本。


---

## 7️⃣ Usage、成本與隱私檢核

| 檢查項目 | 課堂做法 |
|---|---|
| 請求次數 | 每執行一個聊天回合就加 1 |
| Token | 記錄 `response.usage`；不要只看輸出長度 |
| 長對話 | 設定回合上限、最近 N 輪或摘要策略 |
| 保存 | 釐清 `store=True/False` 與自己的資料庫保存政策 |
| 隱私 | 不輸入 API key、個資、未公開成績與機密文件 |
| Reset | 清除目前 session 的 ID、transcript 與 usage |

`previous_response_id` 讓程式比較簡潔，但不代表先前 token 免費，也不取代應用程式的隱私與保存政策。


---

## 8️⃣ Streaming：先看到片段，再取得完整結果

非串流會等待完整回答後一次回傳；Streaming 會以 SSE 事件逐步送回資料。文字介面最常處理：

- `response.created`：串流開始。
- `response.output_text.delta`：新增文字片段，會出現多次。
- `response.completed`：回應完成，可取得 response ID 與 usage。
- `error`：串流失敗，不應更新成功狀態。


In [ ]:
def stream_response(
    user_input,
    previous_response_id=None,
    instructions=CHAT_INSTRUCTIONS,
    on_delta=None,
):
    """
    使用 Streaming 模式即時處理文字 delta，並回傳完整文字與 completed response。

    Streaming 和一般回應最大的差異是：模型輸出會一小段一小段送回來。
    delta 適合即時顯示在畫面上，但它不是最終狀態；只有收到 response.completed，
    才能取得正式 response id 與 usage。因此這個函式會同時保存 chunks 和 completed_response。

    參數 (Args):
        user_input:            使用者本輪輸入。
        previous_response_id:  選填，上一輪 response id，用來延續對話。
        instructions:          本輪要送出的角色與回答規則。
        on_delta:              選填 callback；每收到一段文字 delta 就呼叫一次。

    回傳 (Returns):
        (full_text, completed_response): 完整輸出文字，以及 completed 事件中的最終 response。

    可能拋出 (Raises):
        ValueError: user_input 是空白或非字串。
        RuntimeError: streaming 過程失敗，或沒有收到 response.completed。
    """

    # 情況一：輸入不合法
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("user_input 必須是非空白字串。")

    request = {
        "model": MODEL,
        "instructions": instructions,
        "input": user_input,
        "store": True,
        "stream": True,
    }
    # 情況二：需要延續上一輪對話
    # streaming 也可以使用 previous_response_id；差別只在回應以 event stream 形式回來。
    if previous_response_id:
        request["previous_response_id"] = previous_response_id

    chunks = []
    completed_response = None
    # 沒傳 callback 時，預設直接把 delta 即時印在 notebook 輸出區。
    callback = on_delta or (lambda delta: print(delta, end="", flush=True))

    try:
        for event in client.responses.create(**request):
            # 情況一：收到文字片段
            # delta 是尚未完成的部分輸出，適合即時顯示，但不要在此更新正式狀態。
            if event.type == "response.output_text.delta":
                chunks.append(event.delta)
                callback(event.delta)
            # 情況二：串流正式完成
            # completed 事件包含最終 response id 與 usage；狀態更新應以它為準。
            elif event.type == "response.completed":
                completed_response = event.response
            # 情況三：串流過程回報錯誤
            # 這時不應該把目前累積的 chunks 當成正式回答寫入 transcript。
            elif event.type == "error":
                raise RuntimeError(f"串流事件錯誤：{event}")
    except Exception as exc:
        raise RuntimeError(f"Streaming 失敗：{exc}") from exc

    if completed_response is None:
        raise RuntimeError("串流結束但沒有收到 response.completed。")
    return "".join(chunks), completed_response


### 🧩 為什麼要同時「顯示」與「累積」？

畫面即時顯示只解決 UX；應用程式仍需要完整文字，才能寫入 transcript、保存紀錄、進行後續檢查或交給下一個步驟。因此每個 `delta` 都要顯示並加入 `chunks`，最後再 `"".join(chunks)`。


In [ ]:
streamed_text, streamed_response = stream_response(
    "請用三個短句說明 Streaming 對聊天機器人的好處。"
)
print("\n\n--- 串流完成 ---")
print("完整字數：", len(streamed_text))
# 後續多輪串接要使用 completed response 的 id，而不是中途 delta。
print("Response ID：", streamed_response.id)
if streamed_response.usage:
    print("Usage：", streamed_response.usage)


In [ ]:
class StreamingChatSession(ResponseChatSession):
    """
    在 ResponseChatSession 基礎上加入 Streaming 回答能力。

    這個類別重用父類別的 reset、transcript 與 usage 管理，
    只新增 ask_stream()。重點是：串流過程中可以即時顯示 delta，
    但正式狀態仍要等 completed response 回來後才更新。
    """

    def ask_stream(self, user_text, on_delta=None):
        """
        送出一輪串流聊天，完成後更新 transcript、usage 與 previous_response_id。

        參數 (Args):
            user_text: 使用者本輪輸入。
            on_delta:  選填 callback；每收到一段 delta 時呼叫。

        回傳 (Returns):
            完整的模型回覆文字。
        """

        full_text, response = stream_response(
            user_text,
            previous_response_id=self.previous_response_id,
            instructions=self.instructions,
            on_delta=on_delta,
        )
        # 串流完整完成後，才把本輪加入正式 transcript 與 usage。
        self.previous_response_id = response.id
        self.transcript.extend([
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": full_text},
        ])
        self._add_usage(response)
        return full_text


def run_console_chat(session):
    """
    在 notebook 中啟動簡易 console 聊天迴圈。

    這個函式把 Session 類別包成可互動的小型聊天機器人。
    它支援 /reset 清除本地狀態，/exit 結束迴圈；在 notebook 中若忘記輸入 /exit，
    可以手動中止 cell。這段預設不自動執行，是為了避免課堂 notebook 卡在 input()。

    參數 (Args):
        session: 通常是 StreamingChatSession 物件，需提供 reset() 與 ask_stream()。

    回傳 (Returns):
        無。此函式會持續讀取 input() 並印出串流回覆。
    """

    print("聊天開始：輸入 /reset 重設，輸入 /exit 結束。")
    while True:
        user_text = input("\n你：").strip()
        if user_text == "/exit":
            print("聊天結束。")
            break
        if user_text == "/reset":
            session.reset()
            print("已清除目前對話狀態。")
            continue
        if not user_text:
            print("請輸入內容。")
            continue

        print("AI：", end="", flush=True)
        try:
            session.ask_stream(user_text)
            print()
        except RuntimeError as exc:
            print(f"\n錯誤：{exc}")


streaming_chat = StreamingChatSession()
# 取消下一行註解即可啟動互動聊天；結束時輸入 /exit。
# run_console_chat(streaming_chat)


### 🛑 中斷與錯誤處理

- 串流可能在完成前斷線；不要把半段回答當成成功結果。
- 只有收到 `response.completed` 後，才更新 `previous_response_id` 或對話 history。
- 捕捉 `response.failed`、`error` 與網路例外，向使用者顯示可重試訊息。
- Streaming 的部分輸出比完整回答更難做內容審核；正式應用需設計輸入／輸出審核與中止策略。

---

## 9️⃣ 課堂練習 A（必做）：課程諮詢 Session

建立 `CourseAdvisorSession`：固定課程助教角色、支援兩輪追問、可查看 transcript 與 usage，並能用 `reset()` 開始新對話。這是本週第一個必做練習。

In [ ]:
COURSE_ADVISOR_INSTRUCTIONS = """
# TODO 1：設定課程諮詢助教 Identity 與回答規則
"""


class CourseAdvisorSession(ResponseChatSession):
    """
    練習 A：建立專門回答課程問題的聊天 Session。

    這個類別要重用 ResponseChatSession 的狀態管理能力，
    但改用 COURSE_ADVISOR_INSTRUCTIONS 作為角色與回答規則。
    學生只需要完成初始化流程，不需要重新實作 ask()、reset() 或 usage 累加。
    """

    def __init__(self):
        """
        初始化課程諮詢 Session。

        回傳 (Returns):
            無。完成後物件應具備 previous_response_id、transcript 與 usage。
        """

        # TODO 2：呼叫父類別建構式並傳入 COURSE_ADVISOR_INSTRUCTIONS
        # 提示：父類別會負責 reset、previous_response_id、transcript 與 usage。
        pass


# TODO 3：建立 advisor，完成兩輪追問，印出 transcript 與 usage
# 提示：先問課程內容，再追問剛才的回答，觀察狀態是否延續。


---

## 🔟 課堂練習 B（必做）：串流回覆與完整文字

建立 `ask_with_chunk_count()`：逐段顯示回覆、計算收到幾個 delta，最後回傳完整文字與 response。錯誤或未完成時不可假裝成功。這是本週第二個必做練習。

In [ ]:
def ask_with_chunk_count(user_text, previous_response_id=None):
    """
    練習 B：串流顯示回答，並統計收到多少個文字 delta。

    這個函式要示範 callback 的用途：每次 stream_response() 收到 delta，
    callback 就即時顯示文字並更新計數。學生要注意 chunk_count 是外層變數，
    若在巢狀函式中修改，需要使用 nonlocal。

    參數 (Args):
        user_text:           使用者本輪輸入。
        previous_response_id: 選填，上一輪 response id。

    回傳 (Returns):
        建議回傳完整文字與 response，方便後續統計字數或串接下一輪。
    """

    # TODO 1：建立 chunk_count，用來統計 response.output_text.delta 的數量
    # TODO 2：建立 callback，同時顯示 delta 並累加數量
    # TODO 3：呼叫 stream_response，回傳完整文字與 response
    # 提示：chunk_count 是整數；巢狀 callback 若要修改它，需使用 nonlocal。
    pass


# TODO 4：確認成本後執行一次，印出 delta 數與完整字數


---

## 1️⃣1️⃣ 課堂練習 C（選做）：最近 N 輪＋重要事實

進階選做：使用手動 history 建立 `WindowedMemoryChat`。重要事實放在 instructions；一般對話只保留最近 `max_turns` 輪，避免 history 無限制成長。若課堂時間不足，可作為課後挑戰。

In [ ]:
class WindowedMemoryChat:
    """
    練習 C：同時保存長期重要事實與最近 N 輪對話的聊天類別。

    這個類別不使用 previous_response_id，而是手動組合 input history。
    important_facts 代表長期提示，會放進 instructions；turns 則代表短期對話視窗，
    每次呼叫 API 前只取最近 max_turns 輪，控制送入模型的上下文大小。
    """

    def __init__(self, important_facts=None, max_turns=3):
        """
        初始化長短期記憶狀態。

        參數 (Args):
            important_facts: 長期保留的事實清單，例如使用者偏好或專題主題。
            max_turns:       最多保留幾輪最近對話作為短期視窗。
        """

        # TODO 1：檢查 max_turns，保存重要事實、回合上限與 turns
        # important_facts 是長期提示；turns 是最近對話視窗。
        pass

    def build_instructions(self):
        """
        將長期重要事實整理進 instructions。

        回傳 (Returns):
            包含 CHAT_INSTRUCTIONS 與 important_facts 條列內容的字串。
        """

        # TODO 2：把重要事實加入 instructions
        # 提示：把每個 fact 轉成條列，比直接串成一長句更容易檢查。
        pass

    def build_input(self, user_text):
        """
        建立本輪要送進 Responses API 的手動 history。

        參數 (Args):
            user_text: 本輪新的使用者輸入。

        回傳 (Returns):
            message list，包含最近 max_turns 輪 user/assistant 交錯訊息與本輪 user_text。
        """

        # TODO 3：只取最近 max_turns 輪，展開 user/assistant messages
        # 提示：每一輪 turn 要展開成兩則 message，再加上本次 user_text。
        pass

    def ask(self, user_text):
        """
        使用手動 history 呼叫 API，成功後保存本輪對話。

        參數 (Args):
            user_text: 使用者本輪輸入。

        回傳 (Returns):
            模型回覆文字。
        """

        # TODO 4：拒絕空白輸入；使用手動 history、store=False 呼叫 API，成功後保存新回合
        # 提示：成功收到 response 後再 append，避免 API 失敗時留下半套狀態。
        pass

    def reset_recent_turns(self):
        """
        清除短期對話視窗，但保留長期 important_facts。

        回傳 (Returns):
            無。完成後 turns 應為空，但 important_facts 不變。
        """

        # TODO 5：清除最近對話，但保留 important_facts
        pass


# TODO 6：使用至少兩個 important_facts 建立並測試 memory_chat


### 🔍 Context 管理檢查

先用 `history_size()` 觀察訊息數與總字元數，再搭配回應的 `usage` 判讀實際 token 使用量。接著把 `max_turns` 設成 `1` 執行遺忘實驗：若較早提供的名字已被裁切，模型就不應再可靠地回答。字元數只是方便比較，不等同 token 數或費用。

---

## 1️⃣2️⃣ 聊天機器人測試計畫

至少測試：第一輪、依賴前文的追問、reset 後不可記得舊資料、空白輸入、API 錯誤、串流中斷、長對話與敏感資料提醒。

| 測試 | 預期行為 |
|---|---|
| 先說姓名，再問姓名 | 同一 session 可回答 |
| reset 後再問姓名 | 不應沿用舊 session |
| 傳入空白 | 程式先拒絕，不呼叫 API |
| 串流正常完成 | delta 合併後等於完整顯示內容 |
| 串流失敗 | 不更新 previous response ID |
| 超過保留回合數 | 只送出最近 N 輪與重要事實 |

In [ ]:
chatbot_test_cases = [
    "請用一句話介紹生成式 AI。",
    "請列出剛才回答中的一個關鍵詞。",
]


def run_chatbot_smoke_test():
    """
    付費 smoke test：確認串流聊天 Session 能完成多輪狀態更新。

    smoke test 不是完整單元測試，而是課前快速確認 API key、模型、streaming event、
    Session 狀態更新是否可用。這裡使用 StreamingChatSession.ask_stream()，
    是為了確認 response.completed 回來後才更新 previous_response_id。

    回傳 (Returns):
        無。若狀態沒有成功更新，assert 會中止測試。
    """

    bot = StreamingChatSession()
    for prompt in chatbot_test_cases:
        print(f"\n使用者：{prompt}")
        # 情況一：串流完整成功
        # ask_stream() 內部會等到 completed response 後才更新 previous_response_id。
        bot.ask_stream(prompt)
        # 每輪成功後都應該取得新的 response id，下一輪才能延續對話。
        assert bot.previous_response_id is not None
    print("\n✅ Smoke test 完成")


def run_forgetting_experiment():
    """
    四次付費請求：示範 max_turns=1 如何裁切較早的對話。

    這個實驗讓學生看到「最近對話視窗」和「長期重要事實」的差異。
    max_turns=1 時，build_input() 只會送入最近一輪，因此較早提過的名字可能不在 prompt 裡。

    回傳 (Returns):
        無。此函式會印出每輪回覆與送入模型的 history 規模。
    """

    forgetful = WindowedMemoryChat(max_turns=1)
    for message in ["我叫小華。", "我喜歡吃拉麵。", "今天天氣很好。"]:
        print("使用者：", message)
        print("AI：", forgetful.ask(message))

    # probe_input 可在呼叫 API 前檢查實際送入模型的最近 history。
    probe_input = forgetful.build_input("我叫什麼名字？")
    print("送入模型的 history 規模：", history_size(probe_input))
    print("AI：", forgetful.ask("我叫什麼名字？"))


RUN_FORGETTING_EXPERIMENT = False
if RUN_FORGETTING_EXPERIMENT:
    run_forgetting_experiment()
else:
    print("遺忘實驗尚未執行；確認 4 次 API 成本並完成 WindowedMemoryChat 後，再改為 True。")


RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    run_chatbot_smoke_test()
else:
    print("Smoke test 尚未執行；確認 API 成本並完成類別後，再改為 True。")


## ✅ 完成檢核

- [ ] 能說明無狀態 API 為何不會自動記得上一輪。
- [ ] 能用 `previous_response_id` 或手動 history 完成多輪對話。
- [ ] 能解釋為何 `instructions` 需要在每次請求重新提供。
- [ ] 能辨識 `response.output_text.delta` 與 `response.completed`。
- [ ] 只在串流成功完成後更新對話狀態。
- [ ] 用 `history_size()` 與 `max_turns=1` 實驗觀察 context 裁切。
- [ ] 聊天機器人能處理 `exit`、例外與重設狀態。

## 📝 課後任務：完成個人聊天機器人

選擇「課程 FAQ、學習助教、專題規劃或校園資訊」之一，繳交：

1. 角色與 instructions。
2. 至少三輪、其中一輪依賴前文的對話紀錄。
3. Streaming 顯示與完整文字累積。
4. reset、空白輸入與 API 錯誤處理。
5. 至少五筆測試案例與 usage 紀錄。
6. 150 字說明你的記憶、成本與隱私策略。

<div style="border-left:5px solid #d93025;padding:10px 14px;background:#fce8e6;">
<font color="#b3261e"><b>安全提醒</b></font>：不得提交 API key、真實個資、未公開成績、醫療或財務敏感資料。測試請使用虛構資訊。
</div>


## 📚 延伸閱讀

- OpenAI Conversation state：https://developers.openai.com/api/docs/guides/conversation-state
- OpenAI Streaming responses：https://developers.openai.com/api/docs/guides/streaming-responses
- OpenAI Responses API reference：https://developers.openai.com/api/docs/api-reference/responses

下週將進入：**Structured Outputs、JSON Schema 與資料抽取**。
